# RBPMS / RGC cell counting

This notebook takes you through the whole workflow, top to bottom:

| Step | What happens | Where |
|---|---|---|
| **Inspect** | Reads file headers to check formats, pixel size, and how many tiles each image needs | this notebook |
| **1. Automated count** | Fiji normalizes non-TIFFs to `.tif`, cuts large images into `.tif` tiles, counts each tile, and folds counts per image | this notebook (Fiji runs in the background) |
| **2. Manual review** | You check every field in Fiji and fix the count: add cells it missed, remove ones it got wrong | the Fiji window |
| **3. Analysis** | Tables (.csv) and figures of the reviewed counts, per image, per eye and per animal | this notebook |

**How to use a notebook:** click on a grey code box and press **Shift + Enter** to run it. The result appears below the box,
and the next box is selected. Run the boxes in order. A `[*]` to the left of a box means it is still running.

You only need to change the **Settings** box below.

## Settings
Change the text between the quotes if your folders are somewhere else. Leave everything else as it is.

- `DATA_FOLDER`: the folder with your images, one sub-folder per retina (for example `data/1001L/1001L_20x_a1.czi`).
- `OUTPUT_FOLDER`: where results go. It is created if it does not exist.
- `COUNT_WHICH`: count every retina (`"all"`), only the animals with a given priority in the tracking list
  (for example `"today"`), or a list of animal numbers.
- `TILE_SIZE`: maximum tile core width/height in pixels (`2048` by default; `0` counts each image in one piece).
  Images larger than `TILE_SIZE` are split into overlapping `.tif` tiles and folded back into one row per original image.
- `FIJI_FOLDER`: leave as `None` and the notebook will look for Fiji in the usual places.
  If it cannot find it, type the location, for example `"/Applications/Fiji.app"` on a Mac
  or `r"C:\Users\yourname\Fiji.app"` on Windows (keep the `r` in front).

In [ ]:
DATA_FOLDER = "data"
OUTPUT_FOLDER = "output"
FIJI_FOLDER = None

# Which retinas to count in step 1:
#   "all"      every retina in the images folder
#   "today"    only animals with priority "today" in tracking/animals.csv (any priority word works)
#   ["1001", "1002"]   only these animal numbers
COUNT_WHICH = "all"

# Which eyes the tracking log shows: "L" for left eyes only, "L R" for both
LOG_EYES = "L"

# Tiling: images larger than TILE_SIZE are cut into padded .tif tiles for counting; 0 disables tiling
TILE_SIZE = 2048

# Counting settings, tuned for 20x widefield images at 0.227 micrometres per pixel. See the README before changing.
COUNT_SETTINGS = dict(
    radius=60,           # size of the neighbourhood used to decide what is a cell (pixels)
    sigma=8.0,           # amount of smoothing
    min_diameter=35.0,   # smallest cell body counted (pixels), about 8 micrometres
    max_diameter=130.0,  # largest cell body counted (pixels), about 30 micrometres
    min_contrast=1.15,   # a cell must be this many times brighter than the background around it; 1.0 turns this off
    channel=1,           # which colour channel holds the RGC stain
)

# Analysis settings (step 3)
REVIEWED_ONLY = True     # True: use only images you have reviewed. False: also use automated counts of unreviewed images
SAMPLE_PATTERN = r"^(?P<animal>\d+)(?P<eye>[LR])?"   # reads "1001L" or "1001L - XY" as animal 1001, left eye
FIGURE_FORMAT = "png"    # or "pdf" or "svg" for figures you will edit in Illustrator

In [ ]:
# Set-up: loads the tools this notebook needs (installs them the first time if they are missing)
import sys, importlib
from pathlib import Path
for pkg in ["pandas", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        get_ipython().run_line_magic("pip", f"install -q {pkg}")
sys.path.insert(0, str(Path("scripts").resolve()))
import rbpms
OUTPUT = Path(OUTPUT_FOLDER)
fiji = rbpms.find_fiji(FIJI_FOLDER)
if COUNT_WHICH == "all":
    SELECTED_ANIMALS = None
elif isinstance(COUNT_WHICH, str):
    SELECTED_ANIMALS = rbpms.listed_animals(COUNT_WHICH)
    print(f'Animals with priority "{COUNT_WHICH}":', ", ".join(SELECTED_ANIMALS) or "none")
else:
    SELECTED_ANIMALS = [str(a) for a in COUNT_WHICH]
print("Fiji found at:", fiji)
print("Images folder:", Path(DATA_FOLDER).resolve())

## Tracking log
`tracking/animals.csv` (start from `tracking/animals.example.csv`) is the list of animals to process, with a priority for each (for example `today`).
Edit it in Excel or any spreadsheet program, and save it as CSV.

The box below compares that list with the images folder and the results, and writes the log to
`tracking/STATUS.md` and `tracking/status.csv`. **An eye with no images is marked as not mounted.**
Run it at the start and again at the end. The log is never uploaded to GitHub (the lab repo is public); to share it, keep the tracking folder on CRSP (see the README, "Tracking which animals are done").

In [ ]:
from IPython.display import Markdown, display
log = rbpms.update_log(DATA_FOLDER, OUTPUT_FOLDER, eyes=LOG_EYES)
listed = log[(log.priority != "") | (log.requested != "")]
print(f"{len(listed)} retinas on the list, {len(log) - len(listed)} other retinas in the images folder")
display(listed.drop(columns=["requested"]).style.hide(axis="index").format({"stage": lambda s: s[2:]}))

---
# Inspect inputs

Reads each file's metadata without decoding pixels and shows its detected format, dimensions, pixel size (`um_per_px`),
planned number of tiles, and any `problem` that would prevent counting. Saved to `output/counts/images.csv`.

In [ ]:
survey = rbpms.inspect(DATA_FOLDER, OUTPUT_FOLDER, FIJI_FOLDER,
                       tile_px=TILE_SIZE, animals=SELECTED_ANIMALS, **COUNT_SETTINGS)
display(survey[["sample", "image", "format", "width_px", "height_px", "channels", "z", "um_per_px", "tiles", "problem"]])

---
# Step 1: Automated count

Fiji runs headless in the background without changing your source files in `data/`:
1. Non-`.tif` microscope images (and multi-channel or Z-stack images) are normalized into single-channel 2D `.tif` files under `output/normalized/<retina>/`.
2. Images larger than `TILE_SIZE` are cut into smaller overlapping `.tif` tiles under `output/tiles/<retina>/<image>/`.
3. Simple RGC counts each tile individually, keeping only cells whose centroid sits inside that tile's non-overlapping core.
4. Per-tile counts in `output/counts/tiles.csv` fold into one unified row per original image in `output/counts/auto_counts.csv`.

For every counted field, Fiji saves core-cropped `.tif` and `.jpg` overlays in `output/counted/<retina>/` (plus a full overview `.jpg` for tiled images) and per-cell coordinates in `output/rois/<retina>/`. Tiles already counted are skipped on re-run.

In [ ]:
rbpms.count(DATA_FOLDER, OUTPUT_FOLDER, FIJI_FOLDER,
            animals=SELECTED_ANIMALS, tile_px=TILE_SIZE, **COUNT_SETTINGS)

### Check the automated count
The table shows the number of original images, tiles, and the aggregated automated count for each retina. Below it is one counted image:
**magenta** outlines were counted; **cyan** outlines were found but left out because they were too dim.

If most real cells are cyan, or many outlines sit on background, the settings do not suit these images.
Fix that before reviewing (see the README, "Changing the settings").

In [ ]:
from IPython.display import Image, display
auto_df = rbpms.counts(OUTPUT_FOLDER, level="image")
display(auto_df.groupby("sample").agg(images=("image", "count"), tiles=("tiles", "sum"),
                                      automated_count=("auto_count", "sum"))
               .rename_axis("retina"))
example = sorted((OUTPUT / "counted").rglob("*.jpg"))
if example:
    print("Example:", example[0].relative_to(OUTPUT))
    display(Image(filename=str(example[0]), width=700))

---
# Step 2: Manual review in Fiji

The automated count is a first draft. Check every field and correct it. This is done in the normal Fiji window.
Tiled images are reviewed tile by tile on their core-cropped `.tif`s so no cell can be counted twice across tile borders.

**First time only:** run the box below. It adds the review tool to Fiji's menu. Then close Fiji if it is open, and start it again.

In [ ]:
print("Review tool installed at:", rbpms.install_review_tool(FIJI_FOLDER))
print("Restart Fiji, then choose Plugins > RGC > Review Counts.")

**Reviewing:**
1. Open Fiji and choose **Plugins > RGC > Review Counts**.
2. For the folder, choose this project's `output` folder. Type your initials.
3. Each image opens with a yellow dot on every counted cell. Make it so that **every RGC has exactly one dot**:

| To | Do |
|---|---|
| add a cell that was missed | click on it |
| remove a dot that is not a cell | hold **Alt** (on a Mac: **Option**) and click on the dot |
| move a dot | drag it |
| zoom in or out | press **+** or **-** |
| move around a zoomed image | hold the space bar and drag |

4. Click **Save and next**. Use **Skip this image** to come back to it later, or **Stop reviewing** to finish for now.
   Your work is saved after every field (`counts/tile_manual_counts.csv`), so you can stop and continue another day.

Run the box below any time to see how far you have got.

In [ ]:
progress = rbpms.review_status(OUTPUT_FOLDER)
print(f"{int(progress.reviewed.sum())} of {int(progress.images.sum())} images reviewed")
progress

---
# Step 3: Analysis

Run the boxes below after reviewing. `rbpms.counts()` folds all completed tile reviews into `counts/manual_counts.csv` and joins them with `counts/auto_counts.csv`, then writes to `output/results/`:
- `counts_per_image.csv`: one row per original image: automated count, cells removed and added in review, final count, `area_mm2`, `cells_per_mm2`
- `counts_per_eye.csv`: the total of all images of each retina
- `counts_per_animal.csv`: left eye, right eye and both eyes combined for each animal
- `figures/`: counts per image, counts per animal, and automated vs reviewed counts

Images that are not reviewed yet (or whose tiles are only partially reviewed) are left out and listed (unless `REVIEWED_ONLY = False` in the settings).

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

results = OUTPUT / "results"
figures = results / "figures"
figures.mkdir(parents=True, exist_ok=True)

img = rbpms.counts(OUTPUT_FOLDER, level="image")

# animal and eye from the retina name; a samples.csv (sample,animal,eye) next to this notebook overrides it
parts = img["sample"].str.extract(SAMPLE_PATTERN)
img["animal"] = parts["animal"].fillna(img["sample"])
img["eye"] = parts["eye"].fillna("")
if Path("samples.csv").exists():
    override = pd.read_csv("samples.csv", dtype=str).set_index("sample")
    img["animal"] = img["sample"].map(override["animal"]).fillna(img["animal"])
    img["eye"] = img["sample"].map(override["eye"]).fillna(img["eye"])

status = img.groupby("sample").agg(images=("image", "count"), reviewed=("reviewed", "sum"))
status["not_reviewed"] = status.images - status.reviewed
print(f"{int(img.reviewed.sum())} of {len(img)} images reviewed")

# things worth checking before using the totals
no_animal = sorted(img.loc[~img["sample"].str.match(SAMPLE_PATTERN), "sample"].unique())
no_eye = sorted(img.loc[img["eye"] == "", "sample"].unique())
copies = img.loc[img["image"].str.contains(r"\(\d+\)"), ["sample", "image"]]
same_eye = (img.drop_duplicates("sample").groupby(["animal", "eye"])["sample"]
            .apply(list).loc[lambda s: s.str.len() > 1])
if no_animal: print("No animal number in the name (counted as their own animal):", no_animal)
if no_eye: print("No L/R in the name:", no_eye)
if len(copies): print("Possible duplicate images:", ", ".join(copies["sample"] + "/" + copies["image"]))
if len(same_eye): print("Same animal and eye in more than one folder (summed together per animal):",
                        {f"{a}{e}": v for (a, e), v in same_eye.items()})
status

## Counts per image
`count` is the reviewed count. Images without a review are left out when `REVIEWED_ONLY = True`;
otherwise their automated count is used and `count_source` says so.

In [ ]:
if REVIEWED_ONLY:
    use = img[img.reviewed].copy()
    use["count"] = use["final_count"]
    use["count_source"] = "reviewed"
else:
    use = img.copy()
    use["count"] = use["final_count"].fillna(use["auto_count"])
    use["count_source"] = use["reviewed"].map({True: "reviewed", False: "automated only"})

missing = img[~img.reviewed]
if REVIEWED_ONLY and len(missing):
    print(f"Left out, not reviewed yet ({len(missing)} images):")
    print(missing.groupby("sample").image.apply(lambda s: ", ".join(s.str.extract(r"_([a-z]\d+)$")[0].fillna(s))).to_string())

use["cells_per_mm2"] = use.apply(
    lambda r: round(r["count"] / r["area_mm2"], 1) if pd.notna(r.get("area_mm2")) and r["area_mm2"] > 0 else pd.NA,
    axis=1,
)
keep_cols = ["animal", "eye", "sample", "image", "tiles", "tiles_reviewed", "auto_count", "removed", "added",
             "count", "area_mm2", "cells_per_mm2", "count_source", "reviewer", "reviewed_at", "note"]
per_image = use[[c for c in keep_cols if c in use.columns]].sort_values(["animal", "eye", "image"])
for c in ["tiles", "tiles_reviewed", "removed", "added", "count"]:
    if c in per_image:
        per_image[c] = per_image[c].astype("Int64")
per_image.to_csv(results / "counts_per_image.csv", index=False)
per_image

## Counts per eye and per animal
Totals are sums over the original images of each retina.

In [ ]:
per_eye = (per_image.groupby(["animal", "eye", "sample"])
           .agg(images=("image", "count"), total_count=("count", "sum"),
                mean_per_image=("count", "mean"), sd_per_image=("count", "std"))
           .reset_index())
per_eye["images_in_retina"] = per_eye["sample"].map(status["images"])
per_eye = per_eye.round(1)
per_eye.to_csv(results / "counts_per_eye.csv", index=False)

labels = {"L": "left_eye", "R": "right_eye", "": "eye_not_in_name"}
per_animal = per_eye.assign(eye_label=per_eye["eye"].map(labels)).pivot_table(
    index="animal", columns="eye_label", values=["total_count", "images"], aggfunc="sum")
per_animal.columns = [f"{eye}_{'count' if what == 'total_count' else 'images'}" for what, eye in per_animal.columns]
for eye in ["left_eye", "right_eye"]:
    for what in ["count", "images"]:
        if f"{eye}_{what}" not in per_animal:
            per_animal[f"{eye}_{what}"] = pd.NA
cols = [c for e in ["left_eye", "right_eye", "eye_not_in_name"] for c in (f"{e}_count", f"{e}_images") if c in per_animal]
per_animal = per_animal[cols].astype("Float64").round(0).astype("Int64")
per_animal["all_count"] = per_animal[[c for c in cols if c.endswith("_count")]].sum(axis=1, min_count=1)
per_animal = per_animal.reset_index()
per_animal.to_csv(results / "counts_per_animal.csv", index=False)
display(per_eye)
per_animal

## Figures

In [ ]:
if per_image.empty:
    print("No reviewed images yet, so there is nothing to plot. Review images in step 2 first.")
else:
    order = per_eye.sort_values(["animal", "eye"])["sample"].tolist()
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(order) + 2), 4))
    for i, s in enumerate(order):
        y = per_image.loc[per_image["sample"] == s, "count"].astype(float)
        ax.scatter([i] * len(y), y, s=18, alpha=0.6, color="#2a6fb0")
        ax.hlines(y.mean(), i - 0.3, i + 0.3, color="black", lw=2)
    ax.set_xticks(range(len(order)), order)
    ax.set_ylabel("RGCs per image")
    ax.set_ylim(bottom=0)
    ax.set_title("RGC count per image (dots) and mean per retina (bar)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figures / f"counts_per_image.{FIGURE_FORMAT}", dpi=200)
    plt.show()

    fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(per_animal) + 2), 4))
    x = range(len(per_animal))
    bottom = pd.Series(0.0, index=per_animal.index)
    for col, color, label in [("left_eye_count", "#2a6fb0", "left eye"), ("right_eye_count", "#d1495b", "right eye"),
                              ("eye_not_in_name_count", "#999999", "eye not in name")]:
        if col in per_animal:
            h = pd.to_numeric(per_animal[col], errors="coerce").fillna(0).astype(float)
            if h.sum() > 0:
                ax.bar(x, h, bottom=bottom, color=color, label=label)
                bottom += h
    for xi, tot in zip(x, per_animal["all_count"]):
        if pd.notna(tot):
            ax.annotate(f"{int(tot)}", (xi, tot), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
    ax.set_ylim(0, max(float(bottom.max()), 1) * 1.12)
    ax.set_xticks(list(x), per_animal["animal"])
    ax.set_ylabel("RGCs counted (sum of images)")
    ax.set_title("RGC count per animal")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figures / f"counts_per_animal.{FIGURE_FORMAT}", dpi=200)
    plt.show()

## Automated vs reviewed counts
How much the reviewers changed. Points on the dashed line were not changed.

In [ ]:
rev = img[img.reviewed]
if len(rev):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(rev["final_count"], rev["auto_count"], s=18, alpha=0.7)
    hi = max(rev["final_count"].max(), rev["auto_count"].max()) * 1.08
    ax.plot([0, hi], [0, hi], "k--", lw=1)
    ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.set_xlabel("reviewed count per image")
    ax.set_ylabel("automated count per image")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figures / f"automated_vs_reviewed.{FIGURE_FORMAT}", dpi=200)
    plt.show()
    print(f"Reviewers removed {int(rev.removed.sum())} and added {int(rev.added.sum())} cells "
          f"over {len(rev)} images (automated total {int(rev.auto_count.sum())}, reviewed total {int(rev.final_count.sum())}).")
else:
    print("No images reviewed yet.")

---
# Update the tracking log
Run this at the end of a session so `tracking/STATUS.md` shows where every retina stands.

In [ ]:
log = rbpms.update_log(DATA_FOLDER, OUTPUT_FOLDER, eyes=LOG_EYES)
display(Markdown((Path("tracking") / "STATUS.md").read_text(encoding="utf-8")))